In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from numpy.linalg import inv
from numpy.linalg import pinv
import cv2
import itertools as it
import os


# Spectrum Transfer Model

In [2]:
st_extend_20 = 0
st_extend_21 = 1
st_extend_30 = 2

class SpectrumTransfer:
    
    def __init__(self):
        self.extend_mode = 0
        self.pca_eigenvectors = None
        self.pca_mean = None
        self.M = None
        self.TransM = None
        self.TransMin = 0.0
        self.TransMax = 1.0
        
    def get_extend_color(self, src_rgb):
        
        extend = None

        if self.extend_mode == 0:
            extend = np.array([src_rgb[:,0], src_rgb[:,1], src_rgb[:,2], np.power(src_rgb[:,0],2), np.power(src_rgb[:,1],2), np.power(src_rgb[:,2],2)])
        elif self.extend_mode == 1:
            extend = np.array([src_rgb[:,0], src_rgb[:,1], src_rgb[:,2], np.power(src_rgb[:,0],2), np.power(src_rgb[:,1],2), np.power(src_rgb[:,2],2),src_rgb[:,0]* src_rgb[:,1],src_rgb[:,0]* src_rgb[:,2],src_rgb[:,1]* src_rgb[:,2]])
        elif self.extend_mode == 2:
            extend = np.array([src_rgb[:,0], src_rgb[:,1], src_rgb[:,2], np.power(src_rgb[:,0],2), np.power(src_rgb[:,1],2), np.power(src_rgb[:,2],2),src_rgb[:,0]* src_rgb[:,1],src_rgb[:,0]* src_rgb[:,2],src_rgb[:,1]* src_rgb[:,2], np.power(src_rgb[:,0],3), np.power(src_rgb[:,1],3), np.power(src_rgb[:,2],3)])

        return extend
    
    def prepare(self):
        self.TransM = np.dot(self.M.T, self.pca_eigenvectors)
        return
     
    def train(self, extend_mode, train_spec, train_color, pca_comps=6):
        self.extend_mode = extend_mode
        
        pca = PCA(n_components=pca_comps, whiten=False)
        pca.fit(train_spec)
        
        self.pca_eigenvectors = pca.components_
        self.pca_mean = pca.mean_

        trans_pca_a = pca.transform(train_spec)
        trans_extend_c = self.get_extend_color(train_color)

        self.M = np.dot(trans_pca_a.T, pinv(trans_extend_c))

        self.prepare()
        
        c = np.arange(256)
        all_color = np.array(np.meshgrid(c, c, c)).T.reshape(-1,3)

        self.TransMin = 0.0
        self.TransMax = 1.0
        
#         min_v = 100.0
#         max_v = -100.0

#         split_num = 64
#         batch_size = int(all_color.shape[0] / split_num)

#         for n in range(64):
#             test_c = all_color[n * batch_size: (n+1) *batch_size, :]
#             test_s = self.transfer(test_c)
#             batch_min = np.min(test_s)
#             batch_max = np.max(test_s)

#             min_v = min(min_v, batch_min)
#             max_v = max(max_v, batch_max)

#         self.TransMin = min_v
#         self.TransMax = max_v
        
        return
    
    def save(self, setup_file):
        np.savez(setup_file, a=self.M, b=self.pca_eigenvectors, c=self.pca_mean, d=np.array([self.TransMin,self.TransMax]), e=np.array([self.extend_mode]))
        
    def load(self, setup_file):
        weight = np.load(setup_file)
        
        self.M = weight["a"]
        self.pca_eigenvectors = weight["b"]
        self.pca_mean = weight["c"]
        
#         self.TransMin = weight['d'][0]
#         self.TransMax = weight['d'][1]
        
        self.extend_mode = weight['e'][0]
        
        self.prepare()
        
        return
 
    def transfer1D(self, src_data):
        
        src_data = src_data.astype(np.float32)

        tar_sepc = np.dot(self.get_extend_color(src_data).T, self.TransM) + self.pca_mean
        
        tar_sepc = (tar_sepc - self.TransMin) / (self.TransMax - self.TransMin)

        return tar_sepc
    
    def transfer(self, src_data):
        
        src_data = src_data.astype(np.float32)
        
        src_shape = src_data.shape

        if len(src_shape) >= 3:
            src_data = src_data.reshape(-1, src_shape[-1])

        tar_sepc = np.dot(self.get_extend_color(src_data).T, self.TransM) + self.pca_mean

        if len(src_shape) >= 3:
            tar_sepc = tar_sepc.reshape(src_shape[0:-1] + tuple([self.pca_mean.shape[0]]))
        
        tar_sepc = (tar_sepc - self.TransMin) / (self.TransMax - self.TransMin)
        
        return tar_sepc

# Color Reproduce Model

In [3]:
class ColorReproducer:
    
    def __init__(self): 
        self.light_spec = None
        self.cmf = None
        self.ma = None
        self.d65 = np.array([0.95047, 1.00000000, 1.08883])
        self.is_ignore_ca = False
        
        self.light_spec_x3 = None
        self.cal_fact = None
        self.k = None
        self.white_light_k = None
        
    def prepare(self):
        self.light_spec_x3 = np.array([self.light_spec,self.light_spec,self.light_spec]).T 
        self.cal_fact = self.cmf * self.light_spec_x3
        self.k = 100.0 / sum(self.cal_fact[:,1])
        
#         print(self.k)
        
#         self.k = 1
        
        self.white_light_k = np.dot(self.cmf.T, self.light_spec) * (self.k / 100.0)
        self.cal_fact = self.cal_fact * self.k
 
        return

    def load(self, setup_path):
        self.light_spec = np.loadtxt(os.path.join(setup_path, 'CR_light.txt'))
        self.cmf = np.loadtxt(os.path.join(setup_path, 'CR_cmf.txt'))
        self.ma = np.loadtxt(os.path.join(setup_path, 'CR_ma.txt'))
        
        self.prepare()
        return
        
    def reproduce(self, img_spec, gain = 1.0, is_normalize = False, filter_band = None):
        
        h,w,c = img_spec.shape
        
        img_spec = img_spec.reshape((-1,c));
        
        fact = None
        
        if self.is_ignore_ca:
            fact = self.cal_fact
        else:
            fact = inv(self.ma).dot(np.diag(self.ma.dot(self.d65) / self.ma.dot(self.white_light_k))).dot(self.ma)
            fact = fact.dot(self.cal_fact.T).T 
            
        op_ref_spec = None
        
        if filter_band is not None:

            filter_band_set = filter_band - 380

            mask = np.zeros((w*h, 401))
            
            mask[:,filter_band_set] = 1

            op_ref_spec = img_spec * mask * gain

        else:

            op_ref_spec = img_spec * gain
        
        
        if is_normalize: 
            
            img_rep_xyz = op_ref_spec.dot(fact)

            min_v = img_rep_xyz.min()
            max_v = img_rep_xyz.max()

            img_rep_xyz = (img_rep_xyz - min_v) / (max_v - min_v) * 255.0
            
        else:
            
            img_rep_xyz = op_ref_spec.dot(fact) / 100.0 * 255.0

        max_v = img_rep_xyz.max()
        
#         if max_v > 255.0:
#             img_rep_xyz = img_rep_xyz / max_v * 255.0
            
        img_rep_xyz = img_rep_xyz.reshape(h, w, 3).astype('uint8')
                
        img_rep_rgb = cv2.cvtColor(img_rep_xyz, cv2.COLOR_XYZ2RGB)
        
        return img_rep_rgb

# Function to display band selected

In [4]:
def display_band_selected(filter_band):
    
    filter_band_set = filter_band - 380
    
    x = np.arange(380, 781, 1)

    select_mask = np.zeros((401))

    select_c = 'g'
    unselect_c = 'gray'
    
    select_mask = select_mask + 0.5
    select_mask[filter_band_set] = 1
 
    plt.figure(figsize=(15, 1))
    plt.title('Select band info')
    plt.xlim(380, 781)
    plt.ylim(0, 1.0)
    plt.xticks(np.arange(380, 781, 10))
    plt.yticks([])
    plt.grid(True)
    barlist = plt.bar(x, select_mask, width=1, color=unselect_c)
    
    for i in filter_band_set:
        barlist[i].set_color(select_c)
        
    plt.show()

# Load model

In [5]:
transfer = SpectrumTransfer()
transfer.load("weight.npz")

reproducer = ColorReproducer()
reproducer.load('cr_weights')

# Batch Transfer

In [8]:
def display_band_selected(filter_band):
    filter_band_set = filter_band - 380
    x = np.arange(380, 781, 1)
    select_mask = np.zeros((401))
    select_c = 'g'

    unselect_c = 'gray'
    select_mask = select_mask + 0.5
    select_mask[filter_band_set] = 1
transfer = SpectrumTransfer()
transfer.load("weight.npz")
reproducer = ColorReproducer()
reproducer.load('cr_weights')

image_path = r'C:\Users\sai\Downloads\CIT\Team 8\WL\Serrated\WL'
nbi_path = r'C:\Users\sai\Downloads\CIT\Team 8\Band12\Serrated'
gain = 4
is_normalize = False
filter_band = np.arange(415, 430, 1)
filter_band = np.append(filter_band, np.arange(540, 560, 1))
#filter_band = np.append(filter_band, np.arange(640, 660, 1))

if not os.path.isdir(nbi_path):
    os.makedirs(nbi_path)
for filename in os.listdir(image_path):
    file = os.path.join(image_path, filename)
    name = file.split('.')[0]
    img = cv2.imread(file)
    if img is not None:
        img = cv2.resize(img, (684, 456))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        spec = transfer.transfer(img)
        nbi = reproducer.reproduce(spec, gain, is_normalize, filter_band)
        nbi = cv2.cvtColor(nbi, cv2.COLOR_RGB2BGR)
        tar_file = os.path.join(nbi_path, filename)
        cv2.imwrite(tar_file, nbi)
print("All the images have been converted to HSI images")

All the images have been converted to HSI images


In [ ]:
def display_band_selected(filter_band):
    filter_band_set = filter_band - 380
    x = np.arange(380, 781, 1)
    select_mask = np.zeros((401))
    select_c = 'g'

    unselect_c = 'gray'
    select_mask = select_mask + 0.5
    select_mask[filter_band_set] = 1
transfer = SpectrumTransfer()
transfer.load("weight.npz")
reproducer = ColorReproducer()
reproducer.load('cr_weights')

image_path = r'C:\Users\sai\Downloads\CIT\Team 8\WL\Hyperplastic\WL'
nbi_path = r'C:\Users\sai\Downloads\CIT\Team 8\Band12\Hyperplastic'
gain = 4
is_normalize = False
filter_band = np.arange(415, 430, 1)
filter_band = np.append(filter_band, np.arange(540, 560, 1))
#filter_band = np.append(filter_band, np.arange(640, 660, 1))

if not os.path.isdir(nbi_path):
    os.makedirs(nbi_path)
for filename in os.listdir(image_path):
    file = os.path.join(image_path, filename)
    name = file.split('.')[0]
    img = cv2.imread(file)
    if img is not None:
        img = cv2.resize(img, (684, 456))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        spec = transfer.transfer(img)
        nbi = reproducer.reproduce(spec, gain, is_normalize, filter_band)
        nbi = cv2.cvtColor(nbi, cv2.COLOR_RGB2BGR)
        tar_file = os.path.join(nbi_path, filename)
        cv2.imwrite(tar_file, nbi)
print("All the images have been converted to HSI images")

In [ ]:
def display_band_selected(filter_band):
    filter_band_set = filter_band - 380
    x = np.arange(380, 781, 1)
    select_mask = np.zeros((401))
    select_c = 'g'

    unselect_c = 'gray'
    select_mask = select_mask + 0.5
    select_mask[filter_band_set] = 1
transfer = SpectrumTransfer()
transfer.load("weight.npz")
reproducer = ColorReproducer()
reproducer.load('cr_weights')

image_path = r'C:\Users\sai\Downloads\CIT\Team 8\WL\Adenomatous\WL'
nbi_path = r'C:\Users\sai\Downloads\CIT\Team 8\Band12\Adenomatous'
gain = 4
is_normalize = False
filter_band = np.arange(415, 430, 1)
filter_band = np.append(filter_band, np.arange(540, 560, 1))
#filter_band = np.append(filter_band, np.arange(640, 660, 1))

if not os.path.isdir(nbi_path):
    os.makedirs(nbi_path)
for filename in os.listdir(image_path):
    file = os.path.join(image_path, filename)
    name = file.split('.')[0]
    img = cv2.imread(file)
    if img is not None:
        img = cv2.resize(img, (684, 456))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        spec = transfer.transfer(img)
        nbi = reproducer.reproduce(spec, gain, is_normalize, filter_band)
        nbi = cv2.cvtColor(nbi, cv2.COLOR_RGB2BGR)
        tar_file = os.path.join(nbi_path, filename)
        cv2.imwrite(tar_file, nbi)
print("All the images have been converted to HSI images")

All the images have been converted to HSI images


# Single Transfer

In [ ]:
image_file = r'C:\Users\Arvind\Downloads\skin_cancer_wli-20240815T020033Z-001\skin_cancer_wli\SCC\ISIC_0012762.jpg'
nbi_file = r'C:\Users\Arvind\Downloads\Band\SCC\1.jpg'

gain = 2
is_normalize = False
filter_band = np.arange(465, 495, 1)
#filter_band = np.arange(515, 530, 1)
#filter_band = np.arange(610, 630, 1)
#filter_band = np.arange(680, 720, 1)
#filter_band = np.arange(755, 780, 1)
#filter_band = np.append(filter_band, np.arange(400, 450, 1))
filter_band = np.append(filter_band, np.arange(600, 603, 1))

display_band_selected(filter_band)

In [ ]:
img = cv2.imread(image_file)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

spec = transfer.transfer(img);

nbi = reproducer.reproduce(spec, gain, is_normalize, filter_band)
nbi_bgr = cv2.cvtColor(nbi, cv2.COLOR_RGB2BGR)

cv2.imwrite(nbi_file, nbi_bgr)

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title('Original image')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(nbi)
plt.title('Reproduction image')
plt.show()

In [ ]:
img_org = cv2.imread('result.png')
img_org = cv2.cvtColor(img_org, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 8))
plt.imshow(img_org)
plt.title('Masked Image')
plt.show()